# Tutorial - Legacy converter

## 1. Installation

We use the PyPi package from the [Antares legacy-to-GEMS converter repository](https://github.com/AntaresSimulatorTeam/AntaresLegacyModels-to-GEMS-Converter)

In [33]:
import subprocess

# Install the Antares-to-GEMS converter and GemsPy
subprocess.run(
    ["uv", "pip", "install", "antares-legacy-gems-converter"],
    check=True,
)

Using Python 3.12.3 environment at: /home/gmaistre/Documents/GEMS/GEMS/.venv
Resolved 45 packages in 901ms
Uninstalled 2 packages in 20ms
Installed 9 packages in 29ms
 + antares-craft==0.14.0
 + antares-legacy-gems-converter==0.2.1
 + antares-study-version==1.0.20
 + antares-timeseries-generation==0.1.9
 - gemspy==0.2.0
 + gemspy==0.1.2
 - pandas==3.0.3
 + pandas==2.3.3
 + pyarrow==25.0.1
 + pytz==2026.3.post1
 + tzdata==2026.3


CompletedProcess(args=['uv', 'pip', 'install', 'antares-legacy-gems-converter'], returncode=0)

**Restart the kernel** after running the cell above so the newly installed package is picked up.

## 2. Running converter

### 2.1 Introduction of the sample legacy study

The legacy study which is about to be conveted is composed of two areas (`france`, `germany`) with one thermal plant per country and one wind plant in `france`.

| Element  |Power/unit|  Unit    |  Marginal Price   |
|:----------|:----------:|:----------:|:----------:|
| france_thermal_gas    | 10 MW    | 10 units    | 45€ / MW    |
| france_wind_onshore    | 60 MW    | 1 unit    | ***   |
| germany_thermal_gas    | 30 MW    | 5 units    | 60€ / MW    |


In [34]:
from pathlib import Path
from antares.craft import read_study_local

print("STUDY LOADING")
_cwd = Path.cwd()

# Parse the Antares legacy study into a Study object
study = read_study_local(_cwd / "notebook_legacy")
study_name = study.name
print("\tStudy loaded")

STUDY LOADING
	Study loaded


### 2.2 Set the converter's output folder
The folder of where the converted study will be is set as `tmp/converter_output`.

In [35]:
import shutil

# Set the output directory
output_path = _cwd / "tmp" /"converter_output"
if output_path.exists():
    shutil.rmtree(output_path)

print("\tOutput path set")

	Output path set


### 2.3 Set the library used by the converter
Converter uses the `antares-legacy-models` library for adapting Antares legacy structure elements to GEMS models language.

In [36]:
# Path to the antares_legacy_models.yml defining the GEMS component models that each
# Antares concept (area, thermal, load, hydro …) maps to.
lib_path = (_cwd / "legacy_converter_library.yml").resolve()
print("\tLibrary path resolved")

	Library path resolved


### 2.4 Running the conversion
The configuration is now set, the conversion can start.

In [37]:
from antares_gems_converter.input_converter.src.converter import AntaresStudyConverter
from antares_gems_converter.input_converter.src.logger import Logger

# Set logger
logger = Logger("notebook_test", study.path)

print("\nCONVERSION STARTING")

# Build the converter
# mode="full" : generate a standalone GEMS study written from scratch.
# models_to_convert lists every model type present in notebook_legacy.
converter = AntaresStudyConverter(
    study_input=study,
    logger=logger,
    mode="full",
    output_folder=output_path,
    lib_paths=[str(lib_path)],
    models_to_convert=["load", "wind", "thermal",
                       ],
    )

print("\tConverter set")
# Run the full conversion pipeline and write the result to:
#   output_folder/<study_name>/input/system.yml
converter.process_all()
print("\tConversion succeeded")

print(f"Saved to: {converter.output_system_path}")

2026/09/04 17:14:31 logger,line: 41     INFO | Logger object created successfully.. 
2026/09/04 17:14:31 logger,line: 41     INFO | Logger object created successfully.. 
2026/09/04 17:14:31 logger,line: 41     INFO | Logger object created successfully.. 

CONVERSION STARTING
2026/09/04 17:14:32 converter,line: 100  WARNING | 'area' is not in models_to_convert but conversion mode is 'full': adding it automatically.
2026/09/04 17:14:32 converter,line: 100  WARNING | 'area' is not in models_to_convert but conversion mode is 'full': adding it automatically.
2026/09/04 17:14:32 converter,line: 100  WARNING | 'area' is not in models_to_convert but conversion mode is 'full': adding it automatically.
	Converter set
2026/09/04 17:14:32 converter,line: 483     INFO | Converting components of model load...
2026/09/04 17:14:32 converter,line: 483     INFO | Converting components of model load...
2026/09/04 17:14:32 converter,line: 483     INFO | Converting components of model load...
2026/09/04 17

### 2.5 Creating the scenario builder

The `france_wind` component uses a `scenario-group: wind_group` to allow independent scenario mapping for wind time series. GEMS requires a `modeler-scenariobuilder.dat` file in `data-series/` that maps each Monte Carlo scenario to a column of the data series file. 

The wind data series has 1 column and the study runs 1 Monte Carlo scenario (index 0), so the only needed line is:
```
wind_group, 0 = 1
```

In [38]:
# Create the scenario builder file: map wind_group scenario 0 to column 1 of the data series
scenario_builder_path = output_path / study_name / "input" / "data-series" / "modeler-scenariobuilder.dat"

with scenario_builder_path.open("w") as _f:
    _f.write("wind_group, 0 = 1\n")

print(f"Scenario builder written to: {scenario_builder_path}")

Scenario builder written to: /home/gmaistre/Documents/GEMS/GEMS/doc/examples/notebooks/tutorial-four-antares-legacy/tmp/converter_output/notebook_legacy/input/data-series/modeler-scenariobuilder.dat


## 3. Running the converted study with GemsPy

Likewise in the notebooks about the [unit-commitment](../tutorial-one-unit-commitment/tutorial-unit-commitment.ipynb) and the [investment](../tutorial-three-investment/tutorial-invest.ipynb), the Python GEMS interpreter, GemsPy, is used for running the study in this notebook.

### 3.1 Installation of the required libraries

In [39]:
import subprocess, os
subprocess.run(
    ["uv", "sync", "--group", "doc", "--frozen"],
    check=True, capture_output=True,
    cwd=os.path.abspath(os.path.join(os.getcwd(), "../../.."))
)

from importlib.metadata import version
print(f"gemspy version: {version('gemspy')}")

gemspy version: 0.2.0


### 3.2 Loading and Solving the converted study

In [40]:
from gems_craft.study.folder import load_study
from gems_runner.session.session import SimulationSession
from gems_craft.optim_config.parsing import OptimConfig, TimeScopeConfig, ScenarioScopeConfig

study_dir = output_path / study_name

print("STUDY LOADING")
study = load_study(study_dir)
print("\tStudy loaded")

optim_config = OptimConfig(
    time_scope=TimeScopeConfig(first_time_step=0, last_time_step=167),
    scenario_scope=ScenarioScopeConfig(include=[0]),
)

print("\nSOLVING OPTIMIZATION PROBLEM")
result = SimulationSession(study=study, optim_config=optim_config).run()
print("\tOptimization problem solved")

STUDY LOADING
	Study loaded

SOLVING OPTIMIZATION PROBLEM


/home/gmaistre/Documents/GEMS/GEMS/.venv/lib/python3.12/site-packages/gems_craft/model/resolve_library.py:177: UserWarning: Objective contribution 'objective' has a scenario dimension but no explicit expec() operator. Expectation semantics (average over scenarios) are applied automatically. Add expec() explicitly to suppress this warning.
  _resolve_model(m, current_lib.port_types, current_lib.id)


Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
ERROR:   getOptionIndex: Option "solver_logs" is unknown
LP has 2352 rows; 2352 cols; 5712 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 3e+01]
  Cost    [4e+01, 2e+03]
  Bound   [0e+00, 0e+00]
  RHS     [8e+01, 1e+02]
Presolving model
0 rows, 336 cols, 0 nonzeros 0s
0 rows, 0 cols, 0 nonzeros 0s
Presolve reductions: rows 0(-2352); columns 0(-2352); nonzeros 0(-5712) - Reduced to empty
Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Objective value     :  6.7084963800e+07
P-D objective error :  1.1106185529e-16
HiGHS run time      :          0.01
	Optimization problem solved


/home/gmaistre/Documents/GEMS/GEMS/.venv/lib/python3.12/site-packages/linopy/common.py:306: UserWarning: Coordinates across variables not equal. Perform outer join.
  warn(
/home/gmaistre/Documents/GEMS/GEMS/.venv/lib/python3.12/site-packages/linopy/common.py:306: UserWarning: Coordinates across variables not equal. Perform outer join.
  warn(
/home/gmaistre/Documents/GEMS/GEMS/.venv/lib/python3.12/site-packages/linopy/common.py:306: UserWarning: Coordinates across variables not equal. Perform outer join.
  warn(


We can print the objective value found for this study

In [41]:
obj_value = float(result.data[result.data["output"] == "objective-value"]["value"].iloc[0])
print(f"Objective value: {obj_value}")

Objective value: 67084963.80000001
